# 3A v2 · Vietnamese forest bird calls — focal train → soundscape test

Macro ROC-AUC over 10 species. v2 = v1 data recipe (synthetic soundscapes) + **ImageNet-pretrained CNN backbones** + self-training on the unlabelled test soundscapes + family ensemble. All heavy steps live in the `.py` modules of this folder; this notebook walks through them. See `README.md` for the ablation log.

In [ ]:
import os, sys, json
import numpy as np, torch
sys.path.insert(0, '.')
from common import CACHE, DEVICE, SPECIES, load_cache, read_csvs, train_targets, holdout_split, denoise, stationary_floor, to_features, build_model, macro_auc
import mix
print('device', DEVICE)

## 1. Data
* `train/`: 1,300 loud focal clips (one bird), `public_test/` + `private_test/`: 800 quiet passive-recorder soundscapes with 0-3 species.
* Front-ends: RMS-normalised mel power, 64 mel (SED-CNN) and 128 mel (ImageNet backbones), 8 kHz, n_fft 512, hop 128 → 251 frames.
* Training data = **synthetic soundscapes** (`mix.Mixer`): denoised focal clips (0-3, class-balanced) at SNR −10…+12 dB over real test-clip noise floors with procedural distractors (insects, frog chorus, rain, hum, harmonic arches), or over *real* test clips carrying their soft pseudo-labels (round 2).
* Validation: 20 % focal hold-out (same split as v1). **SV** = v1 synthetic recipe; **RV** (new) = held-out birds over the 200 *real* lowest-activity test clips (real frogs/insects/rain/non-target birds), labels = inserted birds.

In [ ]:
tr, pu, pr = read_csvs()
d = load_cache(128, 512)
print(tr.primary_label.value_counts().to_dict())
print({k: v.shape for k, v in d.items()})
mix.configure(128)
tr_idx, val_idx = holdout_split(tr)
m = mix.Mixer(denoise(d['train'])[tr_idx], train_targets(tr)[tr_idx], stationary_floor(np.concatenate([d['public'], d['private']])), seed=0)
x, y = m.sample(); print('mixture', x.shape, 'labels', [SPECIES[i] for i in np.where(y > 0.5)[0]])

## 2. Model
`TVSED`: torchvision ImageNet backbone (ResNet18 / ResNet34 / EfficientNet-B0; ResNet max-pool removed to keep time resolution) on a 3-channel log-mel image (clip-centred, per-bin-median-removed, per-bin-80th-pct-removed) → frequency mean+max pooling → SED attention head (frame logits; clip logit = mean of attention-pooled and max-frame logits). `BirdSED` (v1 light VGG) kept as a diverse family.

In [ ]:
for cfg in [dict(arch='resnet18', nch=3), dict(arch='efficientnet_b0', nch=3), dict(arch='sed', width=32, nch=2)]:
    net = build_model({**cfg, 'pretrained': False})
    print(cfg['arch'], sum(p.numel() for p in net.parameters()) / 1e6, 'M params')

## 3. Training & inference
AdamW + OneCycle, 40 epochs × 1,600 freshly synthesised mixtures, batch 32, BCE, SpecAugment, **EMA weights** (0.999), 4× circular time-shift TTA.

* Round 1 (hold-out, validated): `A1_r18`, `A2_eb0`, `A0_sed`.
* Pseudo-labels: `ps1 = 0.6·A1_r18 + 0.4·v1` on the 800 test clips.
* Round 2 (all focal clips + real test backgrounds with soft pseudo-labels, p=0.5): `F1_r18_ps`, `F3_r18_ps` (ResNet18, 2 seeds), `FS1_sed`; round 3 with ps2 = 0.5·F1 + 0.3·A1 + 0.2·A2: `F3_r18_ps2`.

Final = prob blend ResNet : EfficientNet : SED = 2 : 1 : 1 → public 0.9738 / private 0.9721 (v1: 0.9636 / 0.9627).

`python solution.py` runs everything (GPU auto-detected); a single run is e.g.:

In [ ]:
# !python train.py --name A1_r18 --arch resnet18 --mels 128 --lr 1e-3 --epochs 40
# !python train.py --name F1_r18_ps --arch resnet18 --mels 128 --lr 1e-3 --epochs 40 --full --pseudo $CACHE_3A/ps1.npy --seed 1
for n in ['A0_sed', 'A1_r18', 'A2_eb0']:
    f = CACHE / f'{n}.json'
    if f.exists():
        r = json.loads(f.read_text()); print(n, 'SV', round(r['sv_tta'], 4), 'RV', round(r['rv_tta'], 4))

## 4. Predict
Weighted probability blend of the final models (weights from the RV family study, see README) → `public_submission.csv`, `private_submission.csv`.

In [ ]:
# !python predict.py FINAL_MODEL_NAMES --weights ...   (see solution.FINAL)
import solution
print(solution.FINAL)